In [2]:
# ============================================================
# CTG FETAL DISTRESS DETECTION
# EXPLAINABLE AI + WHAT-IF ANALYSIS + REVIEW PRIORITY
# ============================================================

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

# ============================================================
# 1. LOAD DATA AND MODEL
# ============================================================

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

rf_model = joblib.load("../models/random_forest.pkl")

class_names = {
    1: "NORMAL",
    2: "SUSPECT",
    3: "PATHOLOGICAL"
}

print("============================================")
print("   CTG EXPLAINABLE AI MODULE")
print("============================================")
print(f"\nTest samples : {len(X_test)}")
print(f"Features     : {X_test.shape[1]}")
print("Model        : Random Forest")
print("Model loaded : OK")


# ============================================================
# 2. SAMPLE SELECTOR
# ============================================================

sample_selector = widgets.IntSlider(
    value=0,
    min=0,
    max=len(X_test) - 1,
    step=1,
    description="CTG Sample:",
    continuous_update=False,
    style={"description_width": "100px"},
    layout=widgets.Layout(width="500px")
)

display(sample_selector)


# ============================================================
# 3. MAIN ANALYSIS FUNCTION
# ============================================================

def analyze_sample(index):

    sample = X_test.iloc[[index]]

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    prediction = int(rf_model.predict(sample)[0])

    probabilities = rf_model.predict_proba(sample)[0]

    confidence = probabilities[prediction - 1] * 100

    predicted_state = class_names[prediction]

    # --------------------------------------------------------
    # Feature importance
    # --------------------------------------------------------

    importance = pd.DataFrame({
        "Feature": X_test.columns,
        "Importance": rf_model.feature_importances_,
        "Value": sample.iloc[0].values
    })

    importance = importance.sort_values(
        "Importance",
        ascending=False
    )

    # --------------------------------------------------------
    # DISPLAY WHY
    # --------------------------------------------------------

    print("\n")
    print("============================================")
    print("          WHY THIS PREDICTION?")
    print("============================================")

    print(f"\nCTG Sample       : {index}")
    print(f"Predicted State  : {predicted_state}")
    print(f"Confidence       : {confidence:.2f}%")

    print("\nTop influential features:")
    print("--------------------------------------------")

    for _, row in importance.head(5).iterrows():

        print(
            f"{row['Feature']:<12}"
            f" Importance: {row['Importance']:.4f}"
            f"   Value: {row['Value']:.2f}"
        )

    # --------------------------------------------------------
    # BAR CHART
    # --------------------------------------------------------

    top_features = importance.head(10).iloc[::-1]

    plt.figure(figsize=(8, 5))

    plt.barh(
        top_features["Feature"],
        top_features["Importance"]
    )

    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")
    plt.title("Top Features Influencing Model Prediction")

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------
    # REVIEW PRIORITY
    # --------------------------------------------------------

    print("\n")
    print("============================================")
    print("             REVIEW PRIORITY")
    print("============================================")

    if prediction == 3:
        priority = "HIGH"
        reason = (
            "Pathological prediction requires "
            "priority clinical review."
        )

    elif prediction == 2:
        priority = "MEDIUM"
        reason = (
            "Suspect prediction requires "
            "additional clinical attention."
        )

    elif confidence < 80:
        priority = "MEDIUM"
        reason = (
            "Model confidence is relatively low."
        )

    else:
        priority = "LOW"
        reason = (
            "Normal prediction with adequate "
            "model confidence."
        )

    print(f"\nPredicted State : {predicted_state}")
    print(f"Confidence      : {confidence:.2f}%")
    print(f"Review Priority : {priority}")

    print(f"\n→ {reason}")

    # --------------------------------------------------------
    # WHAT-IF FEATURE SELECTOR
    # --------------------------------------------------------

    print("\n")
    print("============================================")
    print("              WHAT-IF ANALYSIS")
    print("============================================")

    feature_dropdown = widgets.Dropdown(
        options=list(X_test.columns),
        value=importance.iloc[0]["Feature"],
        description="Feature:",
        style={"description_width": "100px"},
        layout=widgets.Layout(width="500px")
    )

    display(feature_dropdown)

    # --------------------------------------------------------
    # WHAT-IF BUTTON
    # --------------------------------------------------------

    what_if_button = widgets.Button(
        description="Run What-If Analysis",
        button_style="primary",
        layout=widgets.Layout(width="200px")
    )

    display(what_if_button)

    what_if_output = widgets.Output()

    display(what_if_output)

    # --------------------------------------------------------
    # WHAT-IF FUNCTION
    # --------------------------------------------------------

    def run_what_if(button):

        feature = feature_dropdown.value

        original_value = float(
            sample.iloc[0][feature]
        )

        minimum = float(
            X_test[feature].min()
        )

        maximum = float(
            X_test[feature].max()
        )

        # Create a range of hypothetical values
        values = np.linspace(
            minimum,
            maximum,
            100
        )

        results = []

        for value in values:

            modified_sample = sample.copy()

            modified_sample.loc[
                modified_sample.index[0],
                feature
            ] = value

            new_prediction = int(
                rf_model.predict(
                    modified_sample
                )[0]
            )

            results.append(new_prediction)

        # ----------------------------------------------------
        # Find first point where prediction changes
        # ----------------------------------------------------

        changed_value = None
        changed_prediction = None

        for value, result in zip(values, results):

            if result != prediction:

                changed_value = value
                changed_prediction = result
                break

        with what_if_output:

            clear_output(wait=True)

            print("\n--------------------------------------------")
            print("WHAT-IF RESULT")
            print("--------------------------------------------")

            print(
                f"\nFeature selected : {feature}"
            )

            print(
                f"Original value   : "
                f"{original_value:.2f}"
            )

            if changed_value is not None:

                print(
                    f"Hypothetical value: "
                    f"{changed_value:.2f}"
                )

                print(
                    f"\nOriginal prediction : "
                    f"{class_names[prediction]}"
                )

                print(
                    f"New prediction      : "
                    f"{class_names[changed_prediction]}"
                )

                print(
                    "\n🔄 PREDICTION CHANGED"
                )

                print(
                    f"{class_names[prediction]}"
                    f" → "
                    f"{class_names[changed_prediction]}"
                )

            else:

                print(
                    "\nPrediction did not change "
                    "across the tested feature range."
                )

            # ------------------------------------------------
            # Plot What-If Response
            # ------------------------------------------------

            plt.figure(figsize=(8, 4))

            plt.plot(
                values,
                results
            )

            plt.axvline(
                original_value,
                linestyle="--",
                label="Original Value"
            )

            plt.xlabel(feature)
            plt.ylabel("Predicted Class")
            plt.title(
                f"What-If Analysis: {feature}"
            )

            plt.yticks(
                [1, 2, 3],
                [
                    "Normal",
                    "Suspect",
                    "Pathological"
                ]
            )

            plt.legend()
            plt.tight_layout()
            plt.show()

    what_if_button.on_click(run_what_if)


# ============================================================
# 4. ANALYZE BUTTON
# ============================================================

analyze_button = widgets.Button(
    description="Analyze CTG Sample",
    button_style="success",
    layout=widgets.Layout(width="200px")
)

display(analyze_button)

analysis_output = widgets.Output()

display(analysis_output)


# ============================================================
# 5. BUTTON ACTION
# ============================================================

def run_analysis(button):

    with analysis_output:

        clear_output(wait=True)

        analyze_sample(
            sample_selector.value
        )


analyze_button.on_click(run_analysis)


# ============================================================
# 6. FINAL NOTE
# ============================================================

print("\n")
print("Select a CTG sample above and click")
print("'Analyze CTG Sample'.")

print("\nAI provides decision support.")
print(
    "Final clinical interpretation remains "
    "with the clinician."
)

   CTG EXPLAINABLE AI MODULE

Test samples : 424
Features     : 41
Model        : Random Forest
Model loaded : OK


IntSlider(value=0, continuous_update=False, description='CTG Sample:', layout=Layout(width='500px'), max=423, …

Button(button_style='success', description='Analyze CTG Sample', layout=Layout(width='200px'), style=ButtonSty…

Output()



Select a CTG sample above and click
'Analyze CTG Sample'.

AI provides decision support.
Final clinical interpretation remains with the clinician.
